# Session 4 - The same RAG, rebuilt with LangChain

We are not starting a new project. We are standing on top of the one we already
built in sessions 1-3, in this same folder, with the same documents, the same
cleaning functions, the same embedding model, the same FAISS index, the same
prompt and the same local LLM.

The only question this notebook asks is:

> **Which of our own components can we replace with a LangChain abstraction,
> and what exactly changes when we do?**

Everything we wrote still lives in `rag_utils.py`, `rag_store.py` and
`rag_pipeline.py`. We import from those files instead of rewriting them.

## 1. What are we changing?

Here is the pipeline we already have. Every box is code we wrote by hand.

```
data/documents/            load_documents()      <- rag_utils.py
        |                  clean_documents()
        v
list of dicts              chunk_documents()
        |
        v
list of chunks             Embedder (SentenceTransformer)   <- rag_store.py
        |
        v
vectors.npy + chunks.json  VectorStore.build_faiss()
        |
        v
Retriever.retrieve()       cosine similarity + top_k        <- rag_pipeline.py
        |
        v
build_prompt()             SYSTEM_PROMPT + USER_TEMPLATE
        |
        v
generate_answer()          ollama.chat()
        |
        v
answer + sources
```

And here is the same pipeline with LangChain abstractions dropped in. Notice how
many boxes are still ours:

| Stage | Our code (sessions 1-3) | LangChain replacement | Package |
|---|---|---|---|
| Load files | `load_documents()` | *(keep ours)* or `PyMuPDF4LLMLoader` | `langchain-pymupdf4llm` |
| Clean text | `clean_text()` | **nothing - LangChain has no opinion** | - |
| Container | `{"text":..., "metadata":...}` | `Document(page_content=..., metadata=...)` | `langchain-core` |
| Chunk | `chunk_documents()` | `RecursiveCharacterTextSplitter` | `langchain-text-splitters` |
| Embed | `Embedder` (SentenceTransformer) | `HuggingFaceEmbeddings` | `langchain-huggingface` |
| Store + search | `VectorStore` + `IndexFlatIP` | `FAISS` vector store | `langchain-community` |
| Retrieve | `Retriever.retrieve()` | `vectorstore.as_retriever()` | `langchain-core` |
| Prompt | `SYSTEM_PROMPT` / `USER_TEMPLATE` | `ChatPromptTemplate` | `langchain-core` |
| Call the LLM | `ollama.chat()` | `ChatOllama` | `langchain-ollama` |
| Wire it together | `RAGPipeline.ask()` | LCEL: `a \| b \| c` | `langchain-core` |

Two things to notice before we write a single line:

1. **The cleaning column is empty.** LangChain does not clean text for you.
   Our `clean_text()` survives untouched, and that is normal - not a gap we
   need to fill.
2. **The embedding model and the LLM do not change.** `all-MiniLM-L6-v2` is a
   Hugging Face model and `llama3.2` runs in our local Ollama. LangChain only
   wraps them in a standard interface. It does not host them, retrain them, or
   make them better.

## 2. What stays the same

Same folder, same files, same functions. We import them.

In [1]:
# Install once (in this project's venv), then restart the kernel:
#   pip install langchain langchain-text-splitters langchain-huggingface \
#               langchain-ollama langchain-community langchain-pymupdf4llm
#
# faiss-cpu, sentence-transformers, pypdf and ollama are already installed
# from sessions 1-3 - LangChain reuses them rather than replacing them.

import importlib.metadata as meta

for package in ["langchain", "langchain-core", "langchain-text-splitters",
                "langchain-huggingface", "langchain-ollama",
                "langchain-community", "langchain-pymupdf4llm",
                "sentence-transformers", "faiss-cpu", "ollama"]:
    try:
        print(f"{package:<30} {meta.version(package)}")
    except meta.PackageNotFoundError:
        print(f"{package:<30} NOT INSTALLED")

langchain                      1.3.14
langchain-core                 1.5.3
langchain-text-splitters       1.1.2
langchain-huggingface          1.2.2
langchain-ollama               1.1.0
langchain-community            0.4.2
langchain-pymupdf4llm          1.28.0
sentence-transformers          5.7.0
faiss-cpu                      1.15.0
ollama                         0.6.2


In [2]:
from pathlib import Path
from rag_utils import project_paths

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"   # session 2's model
CHAT_MODEL = "llama3.2"                                      # session 3's model
OLLAMA_URL = "http://localhost:11434"                        # still our machine
CHUNK_SIZE = 500                                             # session 1's settings
CHUNK_OVERLAP = 50
TOP_K = 4

paths = project_paths()
DOCUMENTS = paths["documents"]

print("project   ", paths["root"])
print("documents ", DOCUMENTS)
print("storage   ", paths["storage"])
print("files     ", [p.name for p in sorted(DOCUMENTS.iterdir()) if p.is_file()])

project    C:\Users\skhal\Desktop\RAG_From_Scratch\notebooks
documents  C:\Users\skhal\Desktop\RAG_From_Scratch\notebooks\data\documents
storage    C:\Users\skhal\Desktop\RAG_From_Scratch\notebooks\storage
files      ['ai_course.pdf', 'documentation.txt']


In [3]:
# Our loading and cleaning code, imported - not rewritten.
from rag_utils import (load_documents, clean_documents, chunk_documents,
                       chunk_text, preview)

raw_documents = load_documents(DOCUMENTS)
our_documents = clean_documents(raw_documents)

print(f"{len(raw_documents)} raw documents -> {len(our_documents)} cleaned documents")
print()
for doc in our_documents:
    meta_ = doc["metadata"]
    page = f" p{meta_['page']}" if meta_.get("page") else ""
    print(f"  {meta_['source']}{page:<4} {len(doc['text']):>6} chars   {preview(doc['text'], 60)}")

5 raw documents -> 5 cleaned documents

  ai_course.pdf p1    2446 chars   AI Builders Bootcamp - Course Notes These notes cover the co ...
  ai_course.pdf p2    2321 chars   Chapter 3: Neural Networks A neural network is a graph of si ...
  ai_course.pdf p3    2460 chars   Chapter 5: The Attention Mechanism Attention is the operatio ...
  ai_course.pdf p4    2134 chars   Chapter 7: Retrieval Augmented Generation Retrieval augmente ...
  documentation.txt       9363 chars   FAISS - Vector Index Handbook Internal engineering notes ::  ...


That is exactly the output from session 1. Nothing about LangChain has happened
yet, and the hardest part of the pipeline - turning messy PDF text into clean
paragraphs - is already done.

## 3. The LangChain `Document`

Our documents are plain dicts:

```python
{"text": "...", "metadata": {"source": "ai_course.pdf", "page": 3, ...}}
```

LangChain's version of the same idea:

```
Document
+-- page_content   (str)   <- our "text"
+-- metadata       (dict)  <- our "metadata"
```

That is the whole abstraction. It is a container with two fields and a fixed
naming convention. The reason it matters is not the class itself - it is that
*every* LangChain component (splitters, vector stores, retrievers) speaks this
one type, so they can be plugged into each other.

In [4]:
from langchain_core.documents import Document

# Our dicts -> LangChain Documents. One line, no information lost.
documents = [Document(page_content=doc["text"], metadata=doc["metadata"])
             for doc in our_documents]

print(type(documents[0]))
print(len(documents), "Documents")

<class 'langchain_core.documents.base.Document'>
5 Documents


In [5]:
first = documents[0]

print("page_content (first 200 chars):")
print(" ", preview(first.page_content, 200))
print()
print("metadata:")
for key, value in first.metadata.items():
    print(f"  {key:<10} {value!r}")

page_content (first 200 chars):
  AI Builders Bootcamp - Course Notes These notes cover the core ideas behind modern language models: machine learning, deep learning, neural networks, transformers, attention, embeddings and retrieval  ...

metadata:
  source     'ai_course.pdf'
  path       'C:\\Users\\skhal\\Desktop\\RAG_From_Scratch\\notebooks\\data\\documents\\ai_course.pdf'
  page       1
  type       'pdf'


**Our metadata came along untouched.** `source`, `path`, `page` and `type` are
fields *we* invented in session 1. LangChain does not require any particular
keys - it just carries the dict around. That is what will let us cite sources at
the end of this notebook with the exact same code we wrote in session 3.

### 3.1 A loader is just something that produces `Document`s

Our `load_pdf()` uses `pypdf`, which has nothing to do with LangChain.
LangChain also publishes PDF loaders. Both end up in the same place:

```
pypdf / PyMuPDF / anything     ->  Document(page_content, metadata)  ->  the rest of the pipeline
```

Let us look at both, so the point is concrete rather than theoretical.

In [6]:
# APPROACH A - our own loader (pypdf), converted to Documents
from rag_utils import load_pdf

pdf_path = DOCUMENTS / "ai_course.pdf"
ours = load_pdf(pdf_path)
ours_as_documents = [Document(page_content=d["text"], metadata=d["metadata"]) for d in ours]

print("A) our load_pdf + pypdf")
print("   pages     :", len(ours_as_documents))
print("   metadata  :", ours_as_documents[0].metadata)
print("   text      :", preview(ours_as_documents[0].page_content, 180))

A) our load_pdf + pypdf


   pages     : 4
   metadata  : {'source': 'ai_course.pdf', 'path': 'C:\\Users\\skhal\\Desktop\\RAG_From_Scratch\\notebooks\\data\\documents\\ai_course.pdf', 'page': 1, 'type': 'pdf'}
   text      : AI Builders Bootcamp - Course Notes These notes cover the core ideas behind modern language models: machine learning, deep learning, neural networks, transformers, attention, embed ...


In [7]:
# APPROACH B - a current LangChain PDF integration (returns Documents directly)
from langchain_pymupdf4llm import PyMuPDF4LLMLoader

loader = PyMuPDF4LLMLoader(str(pdf_path), mode="page")
langchain_pdf_documents = loader.load()

print("B) PyMuPDF4LLMLoader")
print("   pages     :", len(langchain_pdf_documents))
print("   metadata  :", sorted(langchain_pdf_documents[0].metadata))
print("   text      :", preview(langchain_pdf_documents[0].page_content, 180))

C:\Users\skhal\Desktop\RAG_From_Scratch\ven\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


B) PyMuPDF4LLMLoader
   pages     : 4
   metadata  : ['author', 'creationDate', 'creationdate', 'creator', 'file_path', 'format', 'keywords', 'modDate', 'moddate', 'page', 'producer', 'source', 'subject', 'title', 'total_pages', 'trapped']
   text      : # **AI Builders Bootcamp - Course Notes** These notes cover the core ideas behind modern language models: machine learning, deep learning, neural networks, transformers, attention, ...


What the comparison shows:

* **Approach B gives richer metadata** (`creationdate`, `producer`, `total_pages`, ...)
  and markdown-flavoured text - `# **AI Builders Bootcamp**` instead of a flat line.
  Headings surviving as `#` is genuinely useful, because a markdown-aware splitter
  can then chunk on them.
* **Approach B skips our cleaning.** No unicode normalisation, no de-hyphenation,
  no boilerplate removal - unless we run `clean_text()` on it ourselves.
* Both produce `list[Document]`. Downstream, nothing can tell the difference.

**Decision for this notebook: we keep our own loader.** We already tuned
`clean_text()` against this exact corpus, and throwing that away to use a
LangChain-branded loader would be a downgrade dressed up as progress. This is the
first real lesson of the session:

> Using LangChain does not mean every component must come from LangChain.
> An external library plus a conversion to `Document` is a first-class citizen.

*(If you want the best of both: run `PyMuPDF4LLMLoader`, then pass each
`page_content` through our `clean_text()`. Challenge 9 at the end.)*

## 4. Text splitter

Ours (session 1):

```python
chunks = chunk_documents(documents, chunk_size=500, chunk_overlap=50)
```

`chunk_text()` walks a fixed window of 500 characters forward in steps of 450.
Simple, predictable, and it happily cuts words in half.

LangChain:

```python
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)
```

Same two knobs. The difference is the word *recursive*: it tries a list of
separators in order - `["\n\n", "\n", " ", ""]` - and only falls back to the next
one when a piece is still too big. So it breaks on paragraphs first, then lines,
then spaces, and cuts mid-word only as a last resort.

We are not re-teaching chunking. We are swapping one implementation for another
and looking at what moved.

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    add_start_index=True,     # records where each chunk started in the parent document
)

chunks = splitter.split_documents(documents)
our_chunks = chunk_documents(our_documents, CHUNK_SIZE, CHUNK_OVERLAP)

print(f"our chunk_documents()             -> {len(our_chunks)} chunks")
print(f"RecursiveCharacterTextSplitter    -> {len(chunks)} chunks")

our chunk_documents()             -> 44 chunks
RecursiveCharacterTextSplitter    -> 48 chunks


In [9]:
# Look at the boundaries. This is the whole behavioural difference.
print("=" * 78)
print("OURS - fixed 500-character window (note where it stops)")
print("=" * 78)
for chunk in our_chunks[:3]:
    print(f"[{len(chunk['text']):>3} chars] ...{chunk['text'][-70:]!r}")

print()
print("=" * 78)
print("LANGCHAIN - recursive, prefers paragraph and space boundaries")
print("=" * 78)
for chunk in chunks[:3]:
    print(f"[{len(chunk.page_content):>3} chars] ...{chunk.page_content[-70:]!r}")

OURS - fixed 500-character window (note where it stops)
[500 chars] ...'d of from explicit rules written by a developer. A traditional program'
[500 chars] ...'ucture, for example by clustering similar items together. Reinforcemen'
[499 chars] ...'ion that lowers the loss. This procedure is called gradient descent. -'

LANGCHAIN - recursive, prefers paragraph and space boundaries
[500 chars] ...'d of from explicit rules written by a developer. A traditional program'
[496 chars] ...'cture, for example by clustering similar items together. Reinforcement'
[496 chars] ...'ion that lowers the loss. This procedure is called gradient descent. -'


In [10]:
# The same chunk-size experiment from session 1, run through both implementations.
print(f"{'chunk_size':>11}{'ours':>8}{'langchain':>11}{'ours avg':>11}{'lc avg':>9}")
print("-" * 50)
for size in (200, 500, 1000):
    overlap = size // 10
    mine = chunk_documents(our_documents, size, overlap)
    theirs = RecursiveCharacterTextSplitter(chunk_size=size,
                                            chunk_overlap=overlap).split_documents(documents)
    mine_avg = sum(len(c["text"]) for c in mine) / len(mine)
    theirs_avg = sum(len(c.page_content) for c in theirs) / len(theirs)
    print(f"{size:>11}{len(mine):>8}{len(theirs):>11}{mine_avg:>11.0f}{theirs_avg:>9.0f}")

 chunk_size    ours  langchain   ours avg   lc avg
--------------------------------------------------
        200     105        124        197      160
        500      44         48        470      410
       1000      23         23        892      858


Read the table: LangChain produces slightly **more** chunks that are on average
slightly **shorter** than the requested size. That is the cost of respecting
boundaries - `chunk_size` becomes a maximum rather than an exact length. Our
version hits 500 exactly every time and pays for it by slicing through words.

Neither is "correct". Both are the same idea with a different tie-break rule.

In [11]:
# Metadata survives the split, and start_index is added.
sample = chunks[5]
print("metadata after splitting:")
for key, value in sample.metadata.items():
    print(f"  {key:<12} {value!r}")

print()
print("our equivalent chunk metadata:")
for key, value in our_chunks[5]["metadata"].items():
    print(f"  {key:<12} {value!r}")

metadata after splitting:
  source       'ai_course.pdf'
  path         'C:\\Users\\skhal\\Desktop\\RAG_From_Scratch\\notebooks\\data\\documents\\ai_course.pdf'
  page         1
  type         'pdf'
  start_index  2253

our equivalent chunk metadata:
  source       'ai_course.pdf'
  path         'C:\\Users\\skhal\\Desktop\\RAG_From_Scratch\\notebooks\\data\\documents\\ai_course.pdf'
  page         1
  type         'pdf'
  chunk_index  5
  n_chunks     6
  n_chars      196


Compare the two metadata blocks:

* Both kept `source`, `path`, `page`, `type` - the parent document's metadata is
  **copied onto every chunk**. That is the behaviour we implemented by hand in
  `chunk_documents()` with `metadata = dict(doc["metadata"])`.
* Ours adds `chunk_index`, `n_chunks`, `n_chars` and an `id`. LangChain adds
  `start_index`. Different bookkeeping, same purpose: knowing where a chunk came from.

**What did LangChain replace?** Twenty lines of window arithmetic, and only that.
It did not decide the chunk size, it did not clean the text, and it did not
choose which metadata matters.

## 5. Embeddings, approach A: Hugging Face on its own

This is our session 2 code. There is nothing LangChain about it - it is a
Hugging Face model, downloaded from the Hub, running locally through
`sentence-transformers`.

In [12]:
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer(EMBEDDING_MODEL)

texts = [chunk.page_content for chunk in chunks[:3]]
st_vectors = st_model.encode(texts, normalize_embeddings=True)

print("model      ", EMBEDDING_MODEL)
print("dimension  ", st_model.get_sentence_embedding_dimension())
print("shape      ", st_vectors.shape)
print("first 5    ", st_vectors[0][:5])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4580.34it/s]

model       sentence-transformers/all-MiniLM-L6-v2
dimension   384
shape       (3, 384)
first 5     [-0.02216969 -0.05445351  0.05818075  0.02639492  0.01628737]


C:\Users\skhal\AppData\Local\Temp\ipykernel_30204\2946451351.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("dimension  ", st_model.get_sentence_embedding_dimension())


## 6. Embeddings, approach B: the same model through LangChain

`HuggingFaceEmbeddings` from `langchain-huggingface` does not contain a model.
It *loads sentence-transformers underneath* and exposes two methods:

```
embeddings.embed_documents(list_of_texts) -> list[list[float]]
embeddings.embed_query(one_string)        -> list[float]
```

Why two methods for one model? Because some embedding models are asymmetric -
they want documents and queries prefixed differently. The interface makes room
for that even when, as here, both do the same thing.

In [13]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,                        # the same Hugging Face id
    encode_kwargs={"normalize_embeddings": True},      # session 2's unit-vector trick
)

lc_vectors = embeddings.embed_documents(texts)
query_vector = embeddings.embed_query("What is the attention mechanism?")

print("embed_documents ->", type(lc_vectors), f"{len(lc_vectors)} x {len(lc_vectors[0])}")
print("embed_query     ->", type(query_vector), len(query_vector))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3712.73it/s]

embed_documents -> <class 'list'> 3 x 384
embed_query     -> <class 'list'> 384


In [14]:
# Are these actually the same numbers? Do not take it on faith.
import numpy as np

difference = np.abs(np.array(lc_vectors) - st_vectors).max()
print("largest difference between the two sets of vectors:", difference)
print()
print("sentence-transformers:", st_vectors[0][:4])
print("langchain           :", np.array(lc_vectors[0][:4]))
print()
print("identical" if difference < 1e-6 else "different!")

largest difference between the two sets of vectors: 0.0

sentence-transformers: [-0.02216969 -0.05445351  0.05818075  0.02639492]
langchain           : [-0.02216969 -0.05445351  0.05818075  0.02639492]

identical


Zero difference, because it is **the same model doing the same arithmetic**.

```
sentence-transformers/all-MiniLM-L6-v2      <- the model (Hugging Face)
                |
        HuggingFaceEmbeddings               <- a LangChain wrapper: 2 methods
                |
        anything that accepts an Embeddings  <- vector stores, retrievers, ...
```

LangChain did not create this embedding model, improve it, or speed it up. It
gave it a **standard shape** so that a vector store can accept it without knowing
what is inside. That is the entire value proposition, and it is a real one - but
be precise about what it is.

In [15]:
# 6.1 - "Local" is not a figure of speech. Where do the weights live?
from pathlib import Path
import os

cache = Path(os.environ.get("HF_HOME", Path.home() / ".cache" / "huggingface")) / "hub"
print("Hugging Face cache:", cache)
for entry in sorted(cache.glob("models--*")):
    size_mb = sum(f.stat().st_size for f in entry.rglob("*") if f.is_file()) / 1e6
    print(f"  {entry.name:<55} {size_mb:>8.1f} MB")

print()
print("No API key, no network call at embed time, no data leaving this machine.")
print("Set HF_HUB_OFFLINE=1 and everything above still works, because the")
print("weights are already on disk.")

Hugging Face cache: C:\Users\skhal\.cache\huggingface\hub
  models--BAAI--bge-small-en-v1.5                            134.5 MB
  models--cross-encoder--ms-marco-MiniLM-L-6-v2               91.8 MB
  models--sentence-transformers--all-MiniLM-L6-v2             91.6 MB

No API key, no network call at embed time, no data leaving this machine.
Set HF_HUB_OFFLINE=1 and everything above still works, because the
weights are already on disk.


Worth saying out loud, because the tutorials rarely do: `HuggingFaceEmbeddings`
runs the model **in this Python process, on this CPU**. It is not calling a
LangChain service, and it is not calling Hugging Face's API. Swapping to
`OpenAIEmbeddings` would change that - you would gain quality and lose locality,
privacy and the zero-cost-per-call property.

## 7. Vector store

What we built in session 2 (`rag_store.py`):

```python
store = VectorStore.from_chunks(chunks, embedder)   # embed everything
store.build_faiss()                                 # faiss.IndexFlatIP(384)
store.save(paths["storage"])                        # vectors.npy + chunks.json
```

LangChain, same three steps in one line:

```python
vectorstore = FAISS.from_documents(chunks, embeddings)
```

`from_documents` does exactly what our `from_chunks` did: call the embedding
model on every chunk, build a FAISS index over the result, and keep the chunks
side by side with their vectors so a search can return text instead of row
numbers.

One detail we set explicitly, because it is the difference between matching
session 2 and merely resembling it: our index was `IndexFlatIP` (inner product
= cosine similarity on unit vectors). LangChain's FAISS defaults to
`IndexFlatL2` (euclidean distance). On normalised vectors both rank identically,
but the *scores* look completely different - so we ask for inner product and get
back the same numbers we have been reading all along.

In [16]:
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

vectorstore = FAISS.from_documents(
    chunks,
    embeddings,
    distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT,   # = cosine, like session 2
)

print("index type   ", type(vectorstore.index).__name__)   # IndexFlatIP - ours exactly
print("vectors      ", vectorstore.index.ntotal)
print("dimension    ", vectorstore.index.d)
print("chunks kept  ", len(vectorstore.docstore._dict))

C:\Users\skhal\AppData\Local\Temp\ipykernel_30204\4257631241.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


index type    IndexFlatIP
vectors       48
dimension     384
chunks kept   48


In [17]:
# Same query, same model, same index type -> the scores should match session 2.
from rag_store import Embedder, VectorStore

question = "What is the attention mechanism?"

our_embedder = Embedder(EMBEDDING_MODEL)
our_store = VectorStore.from_chunks(our_chunks, our_embedder, show_progress_bar=False)
our_hits = our_store.search(our_embedder.encode_query(question), top_k=3)

print("OURS (VectorStore.search)")
for hit in our_hits:
    print(f"  {hit['score']:.3f}  {hit['metadata']['source']:<18} {preview(hit['text'], 60)}")

print()
print("LANGCHAIN (similarity_search_with_score)")
for doc, score in vectorstore.similarity_search_with_score(question, k=3):
    print(f"  {score:.3f}  {doc.metadata['source']:<18} {preview(doc.page_content, 60)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4164.87it/s]

OURS (VectorStore.search)
  0.794  ai_course.pdf      Chapter 5: The Attention Mechanism Attention is the operatio ...
  0.496  ai_course.pdf      r; it reads the entire sequence at once and lets every token ...
  0.485  ai_course.pdf      en used as weights to average the value vectors. This is cal ...

LANGCHAIN (similarity_search_with_score)
  0.791  ai_course.pdf      Chapter 5: The Attention Mechanism Attention is the operatio ...
  0.526  ai_course.pdf      used as weights to average the value vectors. This is called ...
  0.497  ai_course.pdf      can track syntax while another tracks long range topic infor ...


Same passages, same order, scores agreeing to two decimals. They are not
*bit*-identical because the two chunkers cut in slightly different places - so
each chunk contains slightly different text and embeds slightly differently.
That residual gap is chunking, not FAISS.

In [18]:
# Persistence: ours wrote vectors.npy + chunks.json; LangChain writes its own pair.
lc_store_dir = paths["storage"] / "faiss_langchain"
vectorstore.save_local(str(lc_store_dir))
print("saved:", [p.name for p in sorted(lc_store_dir.iterdir())])

reloaded = FAISS.load_local(
    str(lc_store_dir),
    embeddings,
    allow_dangerous_deserialization=True,   # index.pkl is a pickle: only load your own
)
print("reloaded:", reloaded.index.ntotal, "vectors")

saved: ['index.faiss', 'index.pkl']
reloaded: 48 vectors


`index.faiss` is the raw FAISS index - the same bytes `faiss.write_index` would
produce. `index.pkl` is a pickle of the chunks and their id mapping, which is why
loading it demands `allow_dangerous_deserialization=True`: unpickling a file
someone else made can execute code. Our session 2 format (`.npy` + `.json`) has
no such hazard and can be read in a text editor. A fair trade to know about, not
a reason to panic.

> **An honest note on the package.** FAISS is imported from
> `langchain-community`, and you will see a deprecation warning: that package is
> being sunset in favour of per-provider packages (`langchain-openai`,
> `langchain-chroma`, `langchain-mongodb`, ...). FAISS has no standalone
> LangChain package today, so this remains the documented import - but it tells
> you something true about this ecosystem: **integrations move.** Which is
> exactly why the rest of your code talks to `VectorStore` and `Retriever`
> instead of to FAISS directly. Section 14 swaps the store out in one line.

## 8. Retriever

Our session 3 retriever:

```python
class Retriever:
    def retrieve(self, query, top_k=4, min_score=0.2):
        query_vector = self.embedder.encode_query(query)     # 1. embed the question
        return self.store.search(query_vector, top_k, min_score)   # 2. search
```

LangChain:

```python
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
docs = retriever.invoke("What is the attention mechanism?")
```

The concept has not moved a millimetre:

```
ours:        question -> encode_query -> FAISS search -> top-k hits (dicts)
langchain:   question -> Retriever    -> VectorStore  -> top-k Documents
```

A retriever is nothing more than **"a thing with `.invoke(query) -> list[Document]`"**.
It happens to be backed by a vector store here; it could be backed by a
keyword search, a SQL query, or a web API, and the rest of the chain would not
notice. That is the point of the abstraction.

In [19]:
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

retrieved = retriever.invoke(question)
print(f"{len(retrieved)} Documents for: {question!r}\n")
for i, doc in enumerate(retrieved, start=1):
    page = f" p{doc.metadata['page']}" if doc.metadata.get("page") else ""
    print(f"[{i}] {doc.metadata['source']}{page}")
    print(f"    {preview(doc.page_content, 150)}")

4 Documents for: 'What is the attention mechanism?'

[1] ai_course.pdf p3
    Chapter 5: The Attention Mechanism Attention is the operation that lets a model decide, for every token, which other tokens matter. Each token is proj ...
[2] ai_course.pdf p3
    used as weights to average the value vectors. This is called scaled dot product attention. Self attention means queries, keys and values all come from ...
[3] ai_course.pdf p3
    can track syntax while another tracks long range topic information, and the results are concatenated and projected back. In a decoder, attention is ca ...
[4] ai_course.pdf p2
    reads the entire sequence at once and lets every token look at every other token. This makes training highly parallel on GPUs and removes the long dep ...


In [20]:
# The k experiment from session 3, unchanged in spirit.
for k in (1, 3, 5):
    docs = vectorstore.as_retriever(search_kwargs={"k": k}).invoke(question)
    labels = [f"{d.metadata['source']}" + (f" p{d.metadata['page']}" if d.metadata.get('page') else "")
              for d in docs]
    print(f"k={k}  ->  {len(docs)} docs  {labels}")

k=1  ->  1 docs  ['ai_course.pdf p3']
k=3  ->  3 docs  ['ai_course.pdf p3', 'ai_course.pdf p3', 'ai_course.pdf p3']
k=5  ->  5 docs  ['ai_course.pdf p3', 'ai_course.pdf p3', 'ai_course.pdf p3', 'ai_course.pdf p2', 'ai_course.pdf p2']


In [21]:
# Do not just read answers - read retrieval. Same questions as session 3's TEST_SET.
from rag_pipeline import TEST_SET

print(f"{'expected source':<20}{'retrieved (k=4)':<40}  question")
print("-" * 100)
for q, keywords, expected_source, should_refuse in TEST_SET:
    docs = retriever.invoke(q)
    sources = sorted({d.metadata["source"] for d in docs})
    mark = "" if expected_source is None else ("  hit" if expected_source in sources else "  MISS")
    print(f"{str(expected_source):<20}{str(sources):<40}{mark}  {q[:40]}")

expected source     retrieved (k=4)                           question
----------------------------------------------------------------------------------------------------
ai_course.pdf       ['ai_course.pdf']                         hit  What is the attention mechanism?
ai_course.pdf       ['ai_course.pdf']                         hit  What is deep learning?
documentation.txt   ['documentation.txt']                     hit  Which index type should I use below 50,0
documentation.txt   ['documentation.txt']                     hit  How do I install FAISS with pip?
documentation.txt   ['documentation.txt']                     hit  What is the maximum value of k on GPU?
None                ['documentation.txt']                     How do I configure a Kubernetes ingress 
None                ['documentation.txt']                     What is the boiling point of mercury?
None                ['ai_course.pdf', 'documentation.txt']    Who won the 2027 Champions League final?


Look at the last three rows. Those are the questions our corpus **cannot**
answer, and the retriever still hands back four passages - it always returns
its best `k`, however bad they are.

In session 3 we handled that with `min_score=0.2`, dropping weak hits so the
pipeline could say "I don't know". Where did that go?

`as_retriever` does have `search_type="similarity_score_threshold"`, but its
score normalisation depends on the distance strategy and is easy to get subtly
wrong. The transparent option is the one we already understand: ask for scores
and filter them ourselves.

In [22]:
MIN_SCORE = 0.2      # session 3's threshold, same corpus, same model

def retrieve_with_threshold(query, k=TOP_K, min_score=MIN_SCORE):
    # Session 3's retrieval rule, using LangChain's store to do the search.
    scored = vectorstore.similarity_search_with_score(query, k=k)
    return [doc for doc, score in scored if score >= min_score]

for q in ["What is the attention mechanism?",
          "How do I install FAISS with pip?",
          "Who won the 2027 Champions League final?"]:
    kept = retrieve_with_threshold(q)
    top = vectorstore.similarity_search_with_score(q, k=1)[0][1]
    print(f"top score {top:.3f} -> kept {len(kept)}/{TOP_K} passages   {q}")

top score 0.791 -> kept 4/4 passages   What is the attention mechanism?
top score 0.609 -> kept 4/4 passages   How do I install FAISS with pip?


top score 0.101 -> kept 0/4 passages   Who won the 2027 Champions League final?

That is the answer to "is LangChain a black box?" in one cell. It is not: the
scores are right there, and any policy we can express in Python is still ours to
write. What we lose by using `as_retriever()` directly is not power, it is the
reminder that a default `k` with no threshold will confidently feed the LLM
garbage.

*(For the record, our threshold function returns plain `list[Document]`, so it
can be dropped into a chain anywhere a retriever goes - see challenge 6.)*

## 9. The LLM, approach A: local Ollama on its own

Session 3, unchanged. `llama3.2` is running in the Ollama server on
`localhost:11434`, and we talk to it with the `ollama` Python package.

In [23]:
import ollama

response = ollama.chat(
    model=CHAT_MODEL,
    messages=[{"role": "system", "content": "Answer in exactly one short sentence."},
              {"role": "user", "content": "What is a vector database?"}],
    options={"temperature": 0.0, "num_predict": 100},
)

print("type    ", type(response))
print("answer  ", response["message"]["content"].strip())

type     <class 'ollama._types.ChatResponse'>
answer   A vector database is a type of data storage system designed to efficiently store, manage, and query large amounts of dense vector data, such as those used in computer vision, natural language processing, and machine learning applications.


## 10. The LLM, approach B: the same model through LangChain

`ChatOllama` from `langchain-ollama` sends the same HTTP request to the same
local server. What changes is the shape of the call and the shape of the reply.

In [24]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model=CHAT_MODEL,          # the same model already pulled on this machine
    base_url=OLLAMA_URL,       # still localhost - LangChain hosts nothing
    temperature=0.0,           # session 3's setting: faithful, not creative
    num_predict=400,
)

reply = llm.invoke([("system", "Answer in exactly one short sentence."),
                    ("human", "What is a vector database?")])

print("type    ", type(reply).__name__)
print("answer  ", reply.content.strip())
print("model   ", reply.response_metadata.get("model"))

type     AIMessage
answer   A vector database is a type of data storage system designed to efficiently store, manage, and query large amounts of dense vector data, such as those used in computer vision, natural language processing, and machine learning applications.
model    llama3.2


Same model, same weights, same 2 GB of parameters on the same disk. The
differences are all interface:

| | Ollama directly | ChatOllama |
|---|---|---|
| Call | `ollama.chat(model=..., messages=[...])` | `llm.invoke([...])` |
| Messages | `{"role": "user", "content": ...}` dicts | `("human", ...)` / `HumanMessage` |
| Reply | `dict` -> `response["message"]["content"]` | `AIMessage` -> `reply.content` |
| Streaming | `stream=True` generator of dicts | `llm.stream(...)` |
| Swap provider | rewrite the call | change the constructor |

```
llama3.2 in local Ollama          <- the model. Yours. Offline. Free.
        |
    ChatOllama                    <- a LangChain wrapper: invoke / stream / batch
        |
    any LangChain chain           <- can now be composed with prompts and parsers
```

Nothing about `ChatOllama` makes the model smarter. It makes it **composable**,
which is what the next two sections need.

## 11. Our prompt -> `ChatPromptTemplate`

We are not redesigning the prompt. Session 3's prompt is good: it pins the model
to the context, gives it an exact refusal string, and demands citations. It lives
in `rag_pipeline.py`, so we **import it** and wrap it.

In [25]:
from rag_pipeline import SYSTEM_PROMPT, USER_TEMPLATE

print("=" * 78)
print(SYSTEM_PROMPT)
print("=" * 78)
print(USER_TEMPLATE)
print("=" * 78)

You are a careful assistant for a technical knowledge base.

Rules:
1. Answer using ONLY the numbered context passages provided by the user.
2. If the context does not contain the answer, reply exactly:
   "I don't know based on the provided documents."
3. Cite the passage number in square brackets after each claim, like [2].
4. Never invent facts, numbers, file names or citations.
5. Be concise: two to five sentences unless the question asks for more.
Context passages:
{context}

Question: {question}

Answer (cite passages as [1], [2], ...):


In [26]:
from langchain_core.prompts import ChatPromptTemplate

# The same two strings, declared as a reusable template.
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", USER_TEMPLATE),      # already contains {context} and {question}
])

print("placeholders the template expects:", prompt.input_variables)

placeholders the template expects: ['context', 'question']


In [27]:
# A template is a function from a dict to messages. Look at what it produces.
rendered = prompt.invoke({"context": "[1] (source: demo.txt)\nFAISS is a similarity search library.",
                          "question": "What is FAISS?"})

for message in rendered.messages:
    print("-" * 78)
    print(type(message).__name__)
    print("-" * 78)
    print(message.content)

------------------------------------------------------------------------------
SystemMessage
------------------------------------------------------------------------------
You are a careful assistant for a technical knowledge base.

Rules:
1. Answer using ONLY the numbered context passages provided by the user.
2. If the context does not contain the answer, reply exactly:
   "I don't know based on the provided documents."
3. Cite the passage number in square brackets after each claim, like [2].
4. Never invent facts, numbers, file names or citations.
5. Be concise: two to five sentences unless the question asks for more.
------------------------------------------------------------------------------
HumanMessage
------------------------------------------------------------------------------
Context passages:
[1] (source: demo.txt)
FAISS is a similarity search library.

Question: What is FAISS?

Answer (cite passages as [1], [2], ...):


Compare with `build_prompt()` from session 3, which returned
`{"system": ..., "user": ...}` - two strings we then had to hand-place into
`ollama.chat(messages=[...])`.

`ChatPromptTemplate` returns typed `SystemMessage` / `HumanMessage` objects that
**any** LangChain chat model accepts. Our `{"system":..., "user":...}` dict only
made sense to our own `generate_answer()`. Same prompt text; the packaging is
what became portable.

## 12. Compose the RAG chain

We now have every piece. Before using the fancy syntax, let us do it by hand,
one variable at a time, so there is nothing left to be mystified by.

The data flow is the one we implemented in `RAGPipeline.ask()`:

```
question (str)
    |
    v  retriever.invoke(question)
list[Document]
    |
    v  format the passages into one numbered block
context (str)
    |
    v  prompt.invoke({"context": ..., "question": ...})
messages
    |
    v  llm.invoke(messages)
AIMessage
    |
    v  .content
answer (str)
```

In [28]:
from rag_pipeline import build_context, format_sources

def docs_to_context(docs):
    # LangChain Documents -> session 3's numbered, source-tagged context block.
    # build_context() already does the formatting AND the character budget; it
    # just wants dicts, so we hand it dicts. Reuse, not rewrite.
    return build_context([{"text": d.page_content, "metadata": d.metadata} for d in docs])


# --- the chain, executed by hand -------------------------------------------
step1_docs = retriever.invoke(question)
step2_context = docs_to_context(step1_docs)
step3_messages = prompt.invoke({"context": step2_context, "question": question})
step4_reply = llm.invoke(step3_messages)
step5_answer = step4_reply.content

print("1. question   :", question)
print("2. documents  :", len(step1_docs), "Documents")
print("3. context    :", len(step2_context), "chars")
print("4. messages   :", [type(m).__name__ for m in step3_messages.messages])
print("5. AIMessage  :", type(step4_reply).__name__)
print()
print("answer:")
print(step5_answer)

1. question   : What is the attention mechanism?
2. documents  : 4 Documents
3. context    : 2138 chars
4. messages   : ['SystemMessage', 'HumanMessage']
5. AIMessage  : AIMessage

answer:
The attention mechanism is the operation that lets a model decide, for every token, which other tokens matter. It does this by projecting each token into three vectors: a query, a key, and a value, and then calculating the relevance of one token to another as the dot product between the query and key vectors [1]. This relevance is used as weights to average the value vectors, resulting in scaled dot product attention [2].


That is a complete LangChain RAG pipeline. Five assignments, no magic.

### The same thing as one composed chain

Every component we just used - the retriever, the prompt, the model - is a
**Runnable**: an object with `.invoke()`, `.stream()` and `.batch()`. Because
they all share that interface, `|` can glue them together, and the output of one
becomes the input of the next. That is LCEL (LangChain Expression Language).

The only line that needs explaining is the dict at the front:

```python
{"context": retriever | docs_to_context, "question": RunnablePassthrough()}
```

Read it as "build the dict that `prompt` is asking for". The chain is invoked
with a single string - the question - and that string is sent to **both** values
in parallel:

* `"context"`: the question goes into `retriever`, out come Documents, which flow
  into `docs_to_context` and become one string. (A plain Python function in a
  chain is fine - LangChain wraps it for you.)
* `"question"`: `RunnablePassthrough()` means "hand the input through unchanged".

The result is `{"context": "...", "question": "..."}` - exactly the two
placeholders `prompt.input_variables` reported in section 11.

In [29]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": retriever | docs_to_context, "question": RunnablePassthrough()}
    | prompt              # dict          -> messages
    | llm                 # messages      -> AIMessage
    | StrOutputParser()   # AIMessage     -> str   (just reaches in for .content)
)

print(rag_chain.invoke(question))

The attention mechanism is the operation that lets a model decide, for every token, which other tokens matter. It does this by projecting each token into three vectors: a query, a key, and a value, and then calculating the relevance of one token to another as the dot product between the query and key vectors [1]. This relevance is used as weights to average the value vectors, resulting in scaled dot product attention [2].


In [30]:
# Because everything is a Runnable, streaming comes for free - no rewrite.
for token in rag_chain.stream("What is deep learning?"):
    print(token, end="", flush=True)
print()

Deep

 learning

 is

 a

 subset

...

 layers

.

 [

3

]

In [31]:
# And so does batching (still one model, one machine - just less boilerplate).
answers = rag_chain.batch(["How do I install FAISS with pip?",
                           "What is the maximum value of k on GPU?"])
for text in answers:
    print("-" * 78)
    print(text)

------------------------------------------------------------------------------
To install FAISS with pip, you can use either the CPU build or the GPU build. For the CPU build, run `pip install faiss-cpu`. For the GPU build, run `pip install faiss-gpu` [2]. Note that you should not install both at the same time.
------------------------------------------------------------------------------
The maximum value of k on GPU is 2048 [1].


In [32]:
# The refusal path still works, because it comes from OUR prompt, not from LangChain.
print(rag_chain.invoke("Who won the 2027 Champions League final?"))

I don't know based on the provided documents.


Note what just happened: the model refused because `SYSTEM_PROMPT` - the one we
wrote in session 3 - told it to. LangChain contributed no grounding, no
guardrail and no hallucination check. **Prompt quality is still entirely on us.**

## 12.1 Source citations

No citation framework needed. The metadata we attached in session 1 rode all the
way through the splitter, the embedding model and FAISS, and it is sitting on the
retrieved Documents.

The one thing `rag_chain` does not give us is those Documents - it returns a
string. So we stop before the `|` and keep both, which is exactly what
`RAGPipeline.ask()` did when it returned `{"answer": ..., "hits": ...}`.

In [33]:
answer_chain = prompt | llm | StrOutputParser()      # the second half, reusable

def ask(query, k=TOP_K):
    # question -> (answer, source Documents). The chain, split so we keep both.
    docs = retriever.invoke(query) if k == TOP_K else \
           vectorstore.as_retriever(search_kwargs={"k": k}).invoke(query)
    answer = answer_chain.invoke({"context": docs_to_context(docs), "question": query})
    return answer, docs


answer, docs = ask("Which index type should I use below 50,000 vectors?")
print(answer)
print()
print("Sources:")
for i, doc in enumerate(docs, start=1):
    page = f", page {doc.metadata['page']}" if doc.metadata.get("page") else ""
    print(f"  [{i}] {doc.metadata['source']}{page}  (chars {doc.metadata.get('start_index')})")

print()
print("format_sources() from session 3, unchanged:")
print(" ", format_sources([{"metadata": d.metadata} for d in docs]))

Based on passage [3] ([3]), for IndexFlatIP, which is suitable below roughly 50,000 vectors.

Sources:
  [1] documentation.txt  (chars 8640)
  [2] documentation.txt  (chars 4825)
  [3] documentation.txt  (chars 3880)
  [4] documentation.txt  (chars 6945)

format_sources() from session 3, unchanged:
  ['documentation.txt']


## 13. Side by side with the from-scratch RAG

The real test of a migration: same questions, same corpus, same model, both
pipelines running in the same kernel.

In [34]:
from rag_pipeline import build_pipeline
import time

scratch_rag = build_pipeline(paths)        # session 3's pipeline, one call

comparison_questions = [
    "What is the attention mechanism?",
    "How do I install FAISS with pip?",
    "Who won the 2027 Champions League final?",
]

for q in comparison_questions:
    print("=" * 78)
    print("Q:", q)
    print("=" * 78)

    started = time.perf_counter()
    scratch_result = scratch_rag.ask(q)
    scratch_time = time.perf_counter() - started

    started = time.perf_counter()
    lc_answer, lc_docs = ask(q)
    lc_time = time.perf_counter() - started

    print(f"FROM SCRATCH ({scratch_time:.1f}s, {len(scratch_result['hits'])} passages)")
    print(" ", scratch_result["answer"].replace("\n", "\n  "))
    print("  sources:", scratch_result["sources"])
    print()
    print(f"LANGCHAIN    ({lc_time:.1f}s, {len(lc_docs)} passages)")
    print(" ", lc_answer.replace("\n", "\n  "))
    print("  sources:", format_sources([{"metadata": d.metadata} for d in lc_docs]))
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3618.23it/s]

Q: What is the attention mechanism?


FROM SCRATCH (1.3s, 4 passages)
  The attention mechanism is the operation that lets a model decide, for every token, which other tokens matter. It does this by projecting each token into three vectors: a query, a key, and a value, and then calculating the relevance of one token to another as the dot product between the query and key vectors [1]. This process allows the model to weigh the importance of different tokens in the input sequence [2].
  sources: ['ai_course.pdf page 3', 'ai_course.pdf page 2']

LANGCHAIN    (3.8s, 4 passages)
  The attention mechanism is the operation that lets a model decide, for every token, which other tokens matter. It does this by projecting each token into three vectors: a query, a key, and a value, and then calculating the relevance of one token to another as the dot product between the query and key vectors [1]. This relevance is used as weights to average the value vectors, resulting in scaled dot product attention [2].
  sources: ['ai_course.pdf pa

FROM SCRATCH (1.4s, 4 passages)
  To install FAISS with pip, you can use either the CPU build or the GPU build, depending on your hardware. For CPU builds, use:
  
  pip install faiss-cpu
  
  For NVIDIA GPUs, use:
  
  pip install faiss-gpu
  
  [4]
  sources: ['documentation.txt']

LANGCHAIN    (1.1s, 4 passages)
  To install FAISS with pip, you can use either the CPU build or the GPU build. For the CPU build, run:
  
  pip install faiss-cpu
  
  For the GPU build, run:
  
  pip install faiss-gpu
  sources: ['documentation.txt']

Q: Who won the 2027 Champions League final?


FROM SCRATCH (0.0s, 0 passages)
  I don't know based on the provided documents.
  sources: []

LANGCHAIN    (0.6s, 4 passages)
  I don't know based on the provided documents.
  sources: ['ai_course.pdf page 4', 'ai_course.pdf page 2', 'documentation.txt', 'ai_course.pdf page 1']



On the two answerable questions the pipelines agree: same sources, same
substance, same latency - retrieval is noise in both, the LLM is the whole
budget. Where the wording differs, it is because the two splitters cut in
different places, so the passages are not byte-identical. A chunking difference,
not a LangChain difference.

**The third question is the interesting one.** Look at the passage counts:

* **From scratch: 0 passages, 0.0 seconds.** `min_score=0.2` rejected everything,
  so `RAGPipeline.ask()` short-circuited and never called the LLM at all.
* **LangChain: 4 passages, a full generation.** `as_retriever(search_kwargs={"k": 4})`
  has no threshold. It handed four irrelevant chunks to the model and we paid for
  the tokens.

Both said "I don't know" - but the LangChain one only did so because **our**
`SYSTEM_PROMPT` tells the model to refuse when the context does not support an
answer. Delete that rule and it has four passages of unrelated text to
hallucinate from.

This is the section 8 warning arriving with an invoice: *a default retriever
always returns `k`.* If you want session 3's refusal behaviour back, put
`retrieve_with_threshold` in the chain instead of `retriever` - it returns
`list[Document]`, so nothing else changes.

## 13.1 So what is LangChain actually doing?

Line them up:

| From scratch (ours) | LangChain | Who does the work? |
|---|---|---|
| `load_documents()`, `clean_text()` | `Document` (+ optional loaders) | **pypdf**, our regexes |
| `chunk_documents()` | `RecursiveCharacterTextSplitter` | LangChain (~20 lines of ours) |
| `Embedder` | `HuggingFaceEmbeddings` | **sentence-transformers** |
| `VectorStore`, `IndexFlatIP` | `FAISS` vector store | **faiss-cpu** |
| `Retriever.retrieve()` | `vectorstore.as_retriever()` | faiss + the wrapper |
| `build_prompt()` | `ChatPromptTemplate` | our prompt text |
| `ollama.chat()` | `ChatOllama` | **Ollama**, on our machine |
| `RAGPipeline.ask()` | LCEL `\|` composition | LangChain |

The bolded column is the honest one: **the heavy lifting is still done by
external libraries and models.** LangChain contributes interfaces and glue.

What we genuinely gained:

* **Standard interfaces** - `Embeddings`, `VectorStore`, `Retriever`,
  `BaseChatModel`. Learn four shapes, get hundreds of integrations.
* **Composition** - `|`, and `stream()` / `batch()` for free on anything composed.
* **Swappability** - section 14, where a one-line change replaces a component.
* **Less plumbing** - `from_documents` for the embed-index-store loop.

What we did **not** get, and no amount of LangChain will provide:

* Better retrieval. Same model, same index, same k.
* Better embeddings. Byte-identical vectors, as section 6 proved.
* Better answers. Same llama3.2, same prompt.
* Fewer hallucinations. Our `SYSTEM_PROMPT` is doing that work.
* Any idea of what a good `chunk_size`, `k` or `min_score` is for *your* corpus.

And what we paid: a dependency tree, version churn (`langchain-community` is
being sunset mid-course), and one layer between us and the FAISS call - which is
survivable precisely because we already know what is under it.

## 14. Component swapping

This is where the interfaces earn their keep. In each experiment below, exactly
one component changes and the rest of the pipeline is reused as-is.

### 14.1 A different embedding model

In [35]:
# Same code as section 6, one string changed.
# (First run downloads the model, ~130 MB, then it is cached locally like MiniLM.
#  "sentence-transformers/all-mpnet-base-v2" is the other usual candidate, but it
#  is ~400 MB and 768-dimensional.)
ALTERNATIVE_EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

alt_embeddings = HuggingFaceEmbeddings(
    model_name=ALTERNATIVE_EMBEDDING_MODEL,
    encode_kwargs={"normalize_embeddings": True},
)

print(f"{EMBEDDING_MODEL:<45} dim {len(embeddings.embed_query('hello'))}")
print(f"{ALTERNATIVE_EMBEDDING_MODEL:<45} dim {len(alt_embeddings.embed_query('hello'))}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3775.64it/s]

sentence-transformers/all-MiniLM-L6-v2        dim 384
BAAI/bge-small-en-v1.5                        dim 384


In [36]:
# Both models are 384-dimensional, so the old index would happily ACCEPT the new
# vectors - and return nonsense, because the two spaces are unrelated. Changing
# the embedding model always means rebuilding the store.
alt_vectorstore = FAISS.from_documents(chunks, alt_embeddings,
                                       distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT)
alt_retriever = alt_vectorstore.as_retriever(search_kwargs={"k": TOP_K})

print(f"{'model':<22}{'score':>7}  top passage")
print("-" * 90)
for name, store in [("all-MiniLM-L6-v2", vectorstore), ("bge-small-en-v1.5", alt_vectorstore)]:
    doc, score = store.similarity_search_with_score(question, k=1)[0]
    print(f"{name:<22}{score:>7.3f}  {preview(doc.page_content, 58)}")

model                   score  top passage
------------------------------------------------------------------------------------------
all-MiniLM-L6-v2        0.791  Chapter 5: The Attention Mechanism Attention is the operat ...
bge-small-en-v1.5       0.819  Chapter 5: The Attention Mechanism Attention is the operat ...


In [37]:
# Proof that "same dimension" does not mean "compatible". Search the MiniLM index
# with a bge query vector: FAISS cannot tell, and answers with total confidence.
import numpy as np

bad_vector = alt_embeddings.embed_query(question)     # bge vector...
bad_hits = vectorstore.similarity_search_with_score_by_vector(bad_vector, k=2)  # ...MiniLM index

print("MiniLM index searched with a bge query vector:")
for doc, score in bad_hits:
    print(f"  {score:.3f}  {preview(doc.page_content, 60)}")
print()
print("No error, no warning. The scores collapsed (0.79 -> 0.17), so every")
print("threshold you tuned is now meaningless; on a corpus larger than 48 chunks")
print("the ranking drifts too. Session 2's rule stands: the query and the")
print("documents must go through the SAME model.")

MiniLM index searched with a bge query vector:
  0.169  Chapter 5: The Attention Mechanism Attention is the operatio ...
  0.148  reads the entire sequence at once and lets every token look  ...

No error, no warning. The scores collapsed (0.79 -> 0.17), so every
threshold you tuned is now meaningless; on a corpus larger than 48 chunks
the ranking drifts too. Session 2's rule stands: the query and the
documents must go through the SAME model.


In [38]:
# Everything downstream is unchanged: same chunks, same prompt, same llm, same code.
alt_chain = (
    {"context": alt_retriever | docs_to_context, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)
print(alt_chain.invoke(question))

The attention mechanism is the operation that lets a model decide, for every token, which other tokens matter. It does this by projecting each token into three vectors: a query, a key, and a value, and then calculating the relevance of one token to another as the dot product between the query and key vectors [1]. This relevance is used as weights to average the value vectors, resulting in scaled dot product attention [2].


```
all-MiniLM-L6-v2   ->  |                 |
                       | Embeddings      |  ->  same VectorStore  ->  same Retriever  ->  same chain
bge-small-en-v1.5  ->  | (2 methods)     |
```

Two things the interface does **not** protect you from:

* **Scores are not comparable across models.** Each lives in its own space; a
  0.79 from one means nothing next to a 0.79 from the other. So a `min_score`
  threshold tuned for MiniLM is meaningless for bge - you have to re-tune it.
* **Nothing checks that the query model matches the document model.** Same
  dimension, no error, wrong answers.

What you compare is *ranking quality on your own questions*, which is exactly
what session 3's evaluation harness is for.

### 14.2 A different vector store

`FAISS` is one implementation of the `VectorStore` interface. Here is another
that ships inside `langchain-core` with nothing to install.

In [39]:
from langchain_core.vectorstores import InMemoryVectorStore

memory_store = InMemoryVectorStore.from_documents(chunks, embeddings)   # <- only line that differs
memory_retriever = memory_store.as_retriever(search_kwargs={"k": TOP_K})

memory_chain = (
    {"context": memory_retriever | docs_to_context, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

print("FAISS     :", [d.metadata["source"] for d in retriever.invoke(question)])
print("InMemory  :", [d.metadata["source"] for d in memory_retriever.invoke(question)])
print()
print(memory_chain.invoke(question))

FAISS     : ['ai_course.pdf', 'ai_course.pdf', 'ai_course.pdf', 'ai_course.pdf']
InMemory  : ['ai_course.pdf', 'ai_course.pdf', 'ai_course.pdf', 'ai_course.pdf']



The attention mechanism is the operation that lets a model decide, for every token, which other tokens matter. It does this by projecting each token into three vectors: a query, a key, and a value, and then calculating the relevance of one token to another as the dot product between the query and key vectors [1]. This relevance is used as weights to average the value vectors, resulting in scaled dot product attention [2].


`InMemoryVectorStore` does a brute-force numpy scan - which is precisely our
`VectorStore.search_numpy()` from session 2. It is a fine choice for 48 chunks
and a terrible one for 5 million; FAISS is the opposite. **The interface is what
let us find that out by editing one line.**

### 14.3 A different LLM

The chain asks one thing of the model: that it be a chat model. Any provider's
package satisfies that, and only the constructor line changes.

In [40]:
# All four of these produce something the SAME chain accepts.

# local, ours, free, offline:
#   from langchain_ollama import ChatOllama
#   llm = ChatOllama(model="llama3.2", temperature=0)

# a bigger local model - one string, no other change:
#   llm = ChatOllama(model="qwen3:8b", temperature=0)

# an API model (needs a key, sends your context to a third party):
#   from langchain_openai import ChatOpenAI
#   llm = ChatOpenAI(model="gpt-5", temperature=0)

#   from langchain_anthropic import ChatAnthropic
#   llm = ChatAnthropic(model="claude-sonnet-5", temperature=0)

# ...and then, unchanged:
#   rag_chain = ({"context": retriever | docs_to_context,
#                 "question": RunnablePassthrough()} | prompt | llm | StrOutputParser())

print("The retriever, the prompt, the parser and the chain do not change.")
print("Only the constructor does - and, if you leave your machine, the privacy")
print("and cost profile of the whole system.")

The retriever, the prompt, the parser and the chain do not change.
Only the constructor does - and, if you leave your machine, the privacy
and cost profile of the whole system.


## 15. Student challenges

Work in this notebook. Each one is a small edit, and every one of them is a
question about your corpus that the framework cannot answer for you.

1. **Chunk size.** Rebuild `chunks` with `chunk_size=200` and again with `1000`,
   rebuild the store, and re-run the retrieval table in section 8. At which size
   does `documentation.txt` stop being found for the FAISS questions? Why?

2. **Embedding model.** Swap in a third model - `intfloat/e5-small-v2` (384-d,
   small) or `sentence-transformers/all-mpnet-base-v2` (768-d, ~400 MB). Does the
   top passage for each `TEST_SET` question change? Is it *better*, or just
   different - and how did you decide? What happens to your `MIN_SCORE`?

3. **Another chat model.** Pull a second model with `ollama pull` and change one
   constructor line. Compare the two answers to
   `"What is the attention mechanism?"`. Did the citations survive?

4. **k.** Run section 12's chain with `k = 1, 4, 8`. Find a question where `k=1`
   is wrong and `k=4` is right, and one where more context makes the answer worse.

5. **Metadata.** Print `doc.metadata` for every retrieved Document. Use
   `start_index` to locate a chunk in the original file and read what came right
   before it. Was the boundary sensible?

6. **The grounding prompt.** Copy `SYSTEM_PROMPT` into this notebook, weaken
   rule 2 (delete the exact refusal string), and re-run the off-topic question.
   Then use `retrieve_with_threshold` from section 8 in your chain instead of
   `retriever`, and compare the two ways of not answering.

7. **Head to head.** Run session 3's `evaluate()` against the LangChain chain.
   You will need a small adapter with an `.ask(question)` method returning
   `{"answer": ..., "hits": ...}` - the hits are your retrieved Documents. Which
   pipeline scores higher, and is the gap chunking or generation?

8. **Name the pieces.** For every line in section 12's `rag_chain`, say which
   package it comes from and whether the actual work happens in LangChain, in an
   external library, or in a model. No peeking at the table in 13.1.

9. **Loaders.** Load `ai_course.pdf` with `PyMuPDF4LLMLoader`, run each page's
   text through our `clean_text()`, and split it with
   `MarkdownHeaderTextSplitter` instead. Do the markdown headings give you better
   chunk boundaries than character counting?

## What to take away

You can now answer, in your own words:

* **What LangChain is** - a set of standard interfaces (`Document`, `Embeddings`,
  `VectorStore`, `Retriever`, prompt templates, chat models) plus a composition
  operator (`|`) and a large catalogue of integrations.
* **Why we use it** - swap a component without rewriting the application; get
  `stream`/`batch` for free; skip the plumbing.
* **What LangChain is not** - not a model, not a database, not a hosting service,
  not a retrieval-quality improvement, and not a substitute for understanding
  chunking, embeddings, thresholds and prompts.
* **How external tools fit in** - a Hugging Face model or a local Ollama model
  keeps doing the work; the LangChain class is a wrapper that makes it
  composable. `external tool + LangChain interface = LangChain-compatible
  component`.

The mental model, end to end:

```
data/documents/            <- ours, unchanged
        |
   clean_text()            <- ours, unchanged (LangChain has no opinion here)
        |
   Document                <- langchain-core: page_content + metadata
        |
   RecursiveCharacterTextSplitter
        |
   all-MiniLM-L6-v2        <- Hugging Face model (external)
        |
   HuggingFaceEmbeddings   <- LangChain interface over it
        |
   FAISS vector store      <- faiss-cpu (external), LangChain interface
        |
   Retriever               <- .invoke(query) -> list[Document]
        |
   ChatPromptTemplate      <- our session 3 prompt, wrapped
        |
   llama3.2 in Ollama      <- local model (external)
        |
   ChatOllama              <- LangChain interface over it
        |
   StrOutputParser -> answer + sources
```

> LangChain did not replace the components we learned in sessions 1-3.
> It gave us a standard way to connect and compose them.